In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, date_format, regexp_extract
import requests
import pandas as pd
import io

spark.sql("USE CATALOG scottish_housing_ws")
spark.sql("USE SCHEMA bronze")

In [0]:
url_early = (
    "https://www.nomisweb.co.uk/api/v01/dataset/NM_162_1.data.csv"
    "?geography=TYPE464"
    "&date=latestMINUS119-latestMINUS60"
    "&gender=0"
    "&age=0"
    "&measure=1"
    "&measures=20100"
    "&select=date_name,geography_name,geography_code,obs_value"
)

url_recent = (
    "https://www.nomisweb.co.uk/api/v01/dataset/NM_162_1.data.csv"
    "?geography=TYPE464"
    "&date=latestMINUS59-latest"
    "&gender=0"
    "&age=0"
    "&measure=1"
    "&measures=20100"
    "&select=date_name,geography_name,geography_code,obs_value"
)

r_early = requests.get(url_early)
r_early.raise_for_status()
df_early = pd.read_csv(io.StringIO(r_early.text))

r_recent = requests.get(url_recent)
r_recent.raise_for_status()
df_recent = pd.read_csv(io.StringIO(r_recent.text))

df_pd = pd.concat([df_early, df_recent], ignore_index=True)

print(f"Early: {df_early.shape}, Recent: {df_recent.shape}, Combined: {df_pd.shape}")
print(df_pd['DATE_NAME'].unique()[[0, -1]])

In [0]:
df_pd_scotland = df_pd[df_pd['GEOGRAPHY_CODE'].str.startswith('S12')]
print(df_pd_scotland.shape)

df_bronze = spark.createDataFrame(df_pd_scotland)

(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze.dwp_claimant_count")
)

In [0]:
# Cell 8 - Sanity check
spark.sql("SELECT * FROM bronze.dwp_claimant_count LIMIT 5").show()
spark.sql("SELECT COUNT(*) FROM bronze.dwp_claimant_count").show()

In [0]:
# bronze.dwp_claimant_count
# Source: NOMIS NM_162_1, two requests to avoid 25,000 row cap
# Filtered to S12 geography codes (32 Scottish council areas) before Spark ingestion
# April 2016 to March 2026